# Exploratory Data Analysis

# Import

In [1]:
import pandas as pd
import numpy as np
import altair as alt

## Data Loading

In [2]:
# read the data
heart_disease = pd.DataFrame(pd.read_csv("../dataset/heart_disease.csv"))
heart_disease['disease_label'] = heart_disease['num'].map({
    0:'No disease',
    1:'Mild',
    2:'Moderate',
    3:'Severe',
    4:'Very severe'
})

heart_disease = heart_disease.drop(columns=['num'])
heart_disease.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,disease_label
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,No disease
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,Moderate
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,Mild
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,No disease
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,No disease


In [3]:
numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']

In [4]:
heart_disease.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             920 non-null    int64  
 1   age            920 non-null    int64  
 2   sex            920 non-null    object 
 3   dataset        920 non-null    object 
 4   cp             920 non-null    object 
 5   trestbps       861 non-null    float64
 6   chol           890 non-null    float64
 7   fbs            830 non-null    object 
 8   restecg        918 non-null    object 
 9   thalch         865 non-null    float64
 10  exang          865 non-null    object 
 11  oldpeak        858 non-null    float64
 12  slope          611 non-null    object 
 13  ca             309 non-null    float64
 14  thal           434 non-null    object 
 15  disease_label  920 non-null    object 
dtypes: float64(5), int64(2), object(9)
memory usage: 115.1+ KB


In [5]:
heart_disease.describe()

,id,age,trestbps,chol,thalch,oldpeak,ca
count,920.000000,920.000000,861.000000,890.000000,865.000000,858.000000,309.000000
mean,460.500000,53.510870,132.132404,199.130337,137.545665,0.878788,0.676375
std,265.725422,9.424685,19.066070,110.780810,25.926276,1.091226,0.935653
min,1.000000,28.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,230.750000,47.000000,120.000000,175.000000,120.000000,0.000000,0.000000
50%,460.500000,54.000000,130.000000,223.000000,140.000000,0.500000,0.000000
75%,690.250000,60.000000,140.000000,268.000000,157.000000,1.500000,1.000000
max,920.000000,77.000000,200.000000,603.000000,202.000000,6.200000,3.000000


In [6]:
heart_disease.isnull().sum()

id                 0
age                0
sex                0
dataset            0
cp                 0
trestbps          59
chol              30
fbs               90
restecg            2
thalch            55
exang             55
oldpeak           62
slope            309
ca               611
thal             486
disease_label      0
dtype: int64

## Distribution of heart disease severity (target variable)

In [7]:
color_scheme = [
    "#2E7D32",  # dark green (no disease)
    "#66BB6A",  # light green
    "#FFC107",  # yellow
    "#FB8C00",  # orange
    "#D32F2F"   # red (most severe)
]

In [8]:
# Distribution of heart disease severity (target variable)
alt.Chart(heart_disease).mark_bar(size=30).encode(
    x=alt.X('disease_label:N', title='Heart Disease Severity', scale=alt.Scale(domain=['No disease', 'Mild', 'Moderate', 'Severe', 'Very severe'])),
    y=alt.Y('count():Q', title='Frequency'),
    color=alt.Color('disease_label:N', scale=alt.Scale(domain=['No disease', 'Mild', 'Moderate', 'Severe', 'Very severe'], range=color_scheme), legend=alt.Legend(title='Heart Disease Severity'))
).properties(
    width=300,
    height=300,
    title='Distribution of Heart Disease Severity'
)

alt.Chart(...)

In [9]:
heart_disease['disease_label'].value_counts()

disease_label
No disease     411
Mild           265
Moderate       109
Severe         107
Very severe     28
Name: count, dtype: int64

Interpretations: 
 - Most patients in the dataset fall into the no-disease or mild categories, while severe cases are relatively rare, leading to an imbalanced distribution across severity levels.

## Scatter Matrix for Numeric Features

In [10]:
# Scatter Matrix for numeric features
numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']
scatter_matrix = alt.Chart(heart_disease).mark_circle(size=40, opacity=0.5).encode(
    x=alt.X(alt.repeat("column"), type="quantitative"),
    y=alt.Y(alt.repeat("row"), type="quantitative"),
    color=alt.Color("disease_label:N", legend=alt.Legend(title="Heart Disease Severity"), scale=alt.Scale(domain=['No disease', 'Mild', 'Moderate', 'Severe', 'Very severe'], range=color_scheme))
).properties(
    width=150,
    height=150
).repeat(
    row=numeric_cols,
    column=numeric_cols
).properties(
    title='Scatter Matrix of Numeric Features'
)

scatter_matrix

alt.RepeatChart(...)

## Heatmap for Correlation of the Numeric Features

In [11]:
# rename columns for better readability in the heatmap
rename_dict = {
    "age": "Age",
    "trestbps": "Resting Blood Pressure",
    "chol": "Cholesterol",
    "thalch": "Max Heart Rate",
    "oldpeak": "ST Depression"
}

heart_disease_renamed = heart_disease[numeric_cols].rename(columns=rename_dict)

# compute correlation
corr = heart_disease_renamed.corr().reset_index().melt('index')
corr.columns = ['var1','var2','correlation']

# heatmap
heatmap = alt.Chart(corr).mark_rect().encode(
    x=alt.X('var1:N', title='Feature'),
    y=alt.Y('var2:N', title='Feature'),
    color=alt.Color(
        'correlation:Q',
        scale=alt.Scale(scheme='redblue', domain=(-1, 1)),
        title='Correlation'
    )
)

# text labels
text = alt.Chart(corr).mark_text(
    size=12
).encode(
    x='var1:N',
    y='var2:N',
    text=alt.Text('correlation:Q', format='.2f'),
    color=alt.condition(
        "abs(datum.correlation) > 0.5",
        alt.value('white'),
        alt.value('black')
    )
)

(heatmap + text).properties(
    width=300,
    height=300,
    title='Correlation Heatmap of Numeric Features'
)

alt.LayerChart(...)

Interpretation:
- The correlation heatmap shows that most variables have weak correlations, suggesting low multicollinearity and making them suitable for predictive modeling. 

- The strongest relationship is a negative correlation between age and maximum heart rate (−0.37), indicating that older patients tend to reach lower heart rates during exercise. There is also a mild positive correlation between age and resting blood pressure (0.24), suggesting that blood pressure tends to increase slightly with age.

## Important Observations from the Scatter Matrix

In [12]:
# Define selections for interactivity
severity_select = alt.selection_point(
    fields=['disease_label'],
    bind='legend'
)

# Define a brush selection for zooming
brush = alt.selection_interval()

In [13]:
base = alt.Chart(heart_disease).mark_circle(size=60).encode(
    color=alt.Color(
        'disease_label:N',
        legend=alt.Legend(title="Heart Disease Severity"),
        scale=alt.Scale(
            domain=['No disease','Mild','Moderate','Severe','Very severe'],
            range=color_scheme
        )
    ),
    opacity=alt.condition(
        severity_select & brush,
        alt.value(0.9),
        alt.value(0)
    )
).add_params(
    severity_select,
    brush
)

# 1 Age vs Max Heart Rate
p1 = base.encode(
    x=alt.X('age', title='Age'),
    y=alt.Y('thalch', title='Maximum Heart Rate'),
).properties(title='Age vs Maximum Heart Rate')

# 2 Max Heart Rate vs ST Depression
p2 = base.encode(
    x=alt.X('thalch', title='Maximum Heart Rate'),
    y=alt.Y('oldpeak', title='ST Depression'),
).properties(title='Maximum Heart Rate vs ST Depression')

# 3 Age vs Resting Blood Pressure
p3 = base.encode(
    x=alt.X('age', title='Age'),
    y=alt.Y('trestbps', title='Resting Blood Pressure'),
).properties(title='Age vs Resting Blood Pressure')

scatter_interactive = p1 | p2 | p3

scatter_interactive

/var/folders/z2/0v9m6dxx6bnbxcv2q29tp4br0000gn/T/ipykernel_82677/2538650843.py:38: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  scatter_interactive = p1 | p2 | p3
/opt/miniconda3/envs/cpsc330/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3701: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  exec(code_obj, self.user_global_ns, self.user_ns)


alt.HConcatChart(...)

In [14]:
scatter_interactive.save('interactive_scatter_matrix.html')

Interpretations: 
- The heatmap shows that most variables have weak correlations, but one notable pattern is the negative relationship between age and maximum heart rate, where older patients tend to reach lower heart rates during exercise.
To explore these relationships further, we built an interactive scatter plot so we can examine how these variables relate to different levels of heart disease severity.

- When we highlight more severe cases, we observe that higher ages are more frequently associated with more severe heart disease conditions.

## Age Distribution by Heart Disease Severity

In [15]:
severity_select = alt.selection_point(
    fields=['disease_label'],
    bind='legend'
)

density = alt.Chart(heart_disease).mark_line().transform_density(
    'age',
    groupby=['disease_label'],
    as_=['age', 'density']
).encode(
    x=alt.X('age:Q', title='Age'),
    y=alt.Y('density:Q', title='Density', stack=None),
    color=alt.Color('disease_label:N', scale=alt.Scale(domain=['No disease', 'Mild', 'Moderate', 'Severe', 'Very severe'], range=color_scheme), legend=alt.Legend(title='Heart Disease Severity')),
    opacity=alt.condition(severity_select, alt.value(0.9), alt.value(0.1))
).add_params(
    severity_select
).properties(
    width=400,
    height=400,
    title='Age Distribution by Heart Disease Severity'
)

density

alt.Chart(...)

In [16]:
density.save('density.html')

Interpretation: 
- To better understand this pattern, we next look at the age distribution across different severity levels, which allows us to compare how age shifts as heart disease becomes more severe.

In [17]:
heart_disease.groupby('disease_label')['sex'].value_counts()

disease_label  sex   
Mild           Male      235
               Female     30
Moderate       Male       99
               Female     10
No disease     Male      267
               Female    144
Severe         Male       99
               Female      8
Very severe    Male       26
               Female      2
Name: count, dtype: int64

- The dataset is highly imbalanced in terms of sex, with significantly more male patients than female patients. Because of this imbalance, comparisons across sex may reflect sampling bias rather than true differences in disease severity.

In [18]:
alt.Chart(heart_disease).mark_bar(size=20).encode(
    x=alt.X(
        'disease_label:N',
        title='Heart Disease Severity',
        scale=alt.Scale(domain=['No disease','Mild','Moderate','Severe','Very severe'])
    ),
    y=alt.Y('count():Q', title='Number of Patients'),
    color=alt.Color(
        'disease_label:N',
        legend=None,
        scale=alt.Scale(domain=['No disease','Mild','Moderate','Severe','Very severe'], range=color_scheme)
    )
).facet(
    column=alt.Column(
        'cp:N',
        title='Chest Pain Type'
    )
)

alt.FacetChart(...)

Interpretation:
- The distribution of severity levels looks relatively similar across chest pain types, suggesting that chest pain type alone may not strongly determine disease severity.